## 1. Load the raw synthetic fraud CSV

After generating the dataset using ChatGPT, we:

1. Copied the CSV output into a file named  
   `synthetic_fraud_dataset.csv`.
2. Placed it inside the `data/` folder of our project.

In this step, we use `pandas.read_csv` to load the dataset and inspect
its basic structure (shape and first few rows).

In [64]:
import pandas as pd

# Path to the CSV generated from ChatGPT output
raw_path = "./data/synthetic_fraud_dataset.csv"

# Read the CSV into a pandas DataFrame
df_raw = pd.read_csv(raw_path)

# Show basic information about size and preview of the data
print("Raw shape (rows, columns):", df_raw.shape)
df_raw.head()

Raw shape (rows, columns): (1200, 12)


,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,is_fraud,transaction_note
0,1,1119,2022-04-21T21:41:41,AU,mobile_app,gaming,547.24,78.0,3,3,0,VPN suspected
1,2,1166,2022-01-31T02:59:48,JP,web,luxury,1069.52,17.0,0,2,0,overnight shipping
2,3,1142,2024-09-23T04:12:18,CN,web,travel,305.73,16.0,4,1,0,NaN
3,4,1060,2024-04-06T03:56:36,SG,pos_terminal,electronics,2096.71,76.0,2,2,0,high value item
4,5,1018,2022-01-20T14:39:36,US,pos_terminal,groceries,663.15,18.0,2,3,0,first time merchant


## 2. Inspect column types and completeness

To understand the raw dataset, we check:

- The inferred **data types** of each column (e.g. string, integer, float).
- How many **non-null values** each column has.

This helps our team see which columns might need cleaning, conversion,
or imputation later.


In [65]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             1200 non-null   int64  
 1   customer_id                1200 non-null   int64  
 2   transaction_datetime       1200 non-null   object 
 3   country                    1200 non-null   object 
 4   channel                    1200 non-null   object 
 5   merchant_category          1150 non-null   object 
 6   amount                     1200 non-null   float64
 7   device_trust_score         1148 non-null   float64
 8   num_txn_24h_customer       1200 non-null   int64  
 9   previous_chargeback_count  1200 non-null   int64  
 10  is_fraud                   1200 non-null   int64  
 11  transaction_note           973 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 112.6+ KB


In [66]:
df_raw.describe(include="all")

,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,is_fraud,transaction_note
count,1200.000000,1200.000000,1200,1200,1200,1150,1200.000000,1148.000000,1200.000000,1200.000000,1200.000000,973
unique,NaN,NaN,1200,8,3,5,NaN,NaN,NaN,NaN,NaN,6
top,NaN,NaN,2022-04-21T21:41:41,US,mobile_app,luxury,NaN,NaN,NaN,NaN,NaN,new device
freq,NaN,NaN,1,212,405,250,NaN,NaN,NaN,NaN,NaN,167
mean,600.500000,1090.850000,NaN,NaN,NaN,NaN,2590.959925,52.662021,3.108333,1.733333,0.067500,NaN
std,346.554469,51.657603,NaN,NaN,NaN,NaN,1636.565518,25.400109,2.898896,1.618973,0.250991,NaN
min,1.000000,1000.000000,NaN,NaN,NaN,NaN,-189.250000,-5.000000,0.000000,0.000000,0.000000,NaN
25%,300.750000,1046.000000,NaN,NaN,NaN,NaN,1300.060000,30.750000,1.000000,0.000000,0.000000,NaN
50%,600.500000,1091.000000,NaN,NaN,NaN,NaN,2520.040000,52.000000,3.000000,2.000000,0.000000,NaN
75%,900.250000,1134.000000,NaN,NaN,NaN,NaN,3753.340000,74.000000,5.000000,3.000000,0.000000,NaN


## 3. Data cleaning

In [67]:
import numpy as np

In [68]:
df_clean = df_raw.copy()

Data imputation

In [69]:
Letak imputation sini  - refer data as df_clean okayyy

SyntaxError: invalid syntax (3355165598.py, line 1)

Data validation

In [70]:
# Check if there is negative value @ amount
negative_amount = df_clean['amount'] < 0

print("Count of -ve value in amount transaction",(negative_amount).sum())

print("Rows with negative amount (before fix):")
df_clean.loc[negative_amount]


Count of -ve value in amount transaction 6
Rows with negative amount (before fix):


,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,is_fraud,transaction_note
266,267,1083,2024-10-09T20:42:03,SG,web,travel,-30.20,29.0,2,3,0,overnight shipping
686,687,1072,2024-09-08T06:51:17,UK,mobile_app,groceries,-130.39,82.0,3,3,0,high value item
689,690,1052,2024-03-22T02:52:26,JP,pos_terminal,travel,-186.97,95.0,5,2,0,NaN
1031,1032,1047,2023-05-22T00:07:45,US,mobile_app,travel,-189.25,73.0,5,3,0,new device
1038,1039,1093,2022-10-18T02:20:03,MY,mobile_app,gaming,-185.62,37.0,0,2,0,NaN
1139,1140,1039,2022-12-25T08:55:19,US,pos_terminal,travel,-160.53,35.0,3,10,0,overnight shipping


In [71]:
# Apply absolute value to -ve amount
df_clean.loc[negative_amount, 'amount'] = df_clean.loc[negative_amount, 'amount'].abs()

print("Rows after applying absolute value:")
df_clean.loc[negative_amount]

Rows after applying absolute value:


,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,is_fraud,transaction_note
266,267,1083,2024-10-09T20:42:03,SG,web,travel,30.20,29.0,2,3,0,overnight shipping
686,687,1072,2024-09-08T06:51:17,UK,mobile_app,groceries,130.39,82.0,3,3,0,high value item
689,690,1052,2024-03-22T02:52:26,JP,pos_terminal,travel,186.97,95.0,5,2,0,NaN
1031,1032,1047,2023-05-22T00:07:45,US,mobile_app,travel,189.25,73.0,5,3,0,new device
1038,1039,1093,2022-10-18T02:20:03,MY,mobile_app,gaming,185.62,37.0,0,2,0,NaN
1139,1140,1039,2022-12-25T08:55:19,US,pos_terminal,travel,160.53,35.0,3,10,0,overnight shipping


In [72]:
# Check if device trust score outside [0,100] 
invalid_score = (df_clean['device_trust_score'] < 0) | (df_clean['device_trust_score'] > 100)
print("Count of invalid device_trust_score range(0-100):", ((invalid_score).sum()))

df_clean.loc[invalid_score]

Count of invalid device_trust_score range(0-100): 6


,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,is_fraud,transaction_note
45,46,1032,2023-02-25T02:53:48,AU,pos_terminal,NaN,1165.76,-5.0,1,1,0,NaN
280,281,1114,2022-02-08T15:09:41,SG,web,luxury,3500.58,120.0,6,3,0,NaN
449,450,1136,2024-11-17T02:10:28,SG,mobile_app,travel,2004.84,120.0,1,0,0,VPN suspected
479,480,1101,2022-01-16T11:50:14,JP,mobile_app,gaming,4513.46,105.0,0,1,0,first time merchant
749,750,1169,2023-02-17T20:09:13,JP,web,groceries,4745.77,105.0,1,0,0,new device
827,828,1151,2023-02-01T22:54:23,AU,pos_terminal,luxury,331.02,-5.0,5,2,0,VPN suspected


In [73]:
# Clip values to the max or min range
df_clean.loc[invalid_score, 'device_trust_score'] = df_clean.loc[invalid_score, 'device_trust_score'].clip(0, 100)

print("Rows after applying absolute value:")
df_clean.loc[invalid_score]

Rows after applying absolute value:


,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,is_fraud,transaction_note
45,46,1032,2023-02-25T02:53:48,AU,pos_terminal,NaN,1165.76,0.0,1,1,0,NaN
280,281,1114,2022-02-08T15:09:41,SG,web,luxury,3500.58,100.0,6,3,0,NaN
449,450,1136,2024-11-17T02:10:28,SG,mobile_app,travel,2004.84,100.0,1,0,0,VPN suspected
479,480,1101,2022-01-16T11:50:14,JP,mobile_app,gaming,4513.46,100.0,0,1,0,first time merchant
749,750,1169,2023-02-17T20:09:13,JP,web,groceries,4745.77,100.0,1,0,0,new device
827,828,1151,2023-02-01T22:54:23,AU,pos_terminal,luxury,331.02,0.0,5,2,0,VPN suspected


In [74]:
print(df_clean.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             1200 non-null   int64  
 1   customer_id                1200 non-null   int64  
 2   transaction_datetime       1200 non-null   object 
 3   country                    1200 non-null   object 
 4   channel                    1200 non-null   object 
 5   merchant_category          1150 non-null   object 
 6   amount                     1200 non-null   float64
 7   device_trust_score         1148 non-null   float64
 8   num_txn_24h_customer       1200 non-null   int64  
 9   previous_chargeback_count  1200 non-null   int64  
 10  is_fraud                   1200 non-null   int64  
 11  transaction_note           973 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 112.6+ KB
None


In [75]:
df_clean.describe(include="all")

,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,is_fraud,transaction_note
count,1200.000000,1200.000000,1200,1200,1200,1150,1200.000000,1148.000000,1200.000000,1200.000000,1200.000000,973
unique,NaN,NaN,1200,8,3,5,NaN,NaN,NaN,NaN,NaN,6
top,NaN,NaN,2022-04-21T21:41:41,US,mobile_app,luxury,NaN,NaN,NaN,NaN,NaN,new device
freq,NaN,NaN,1,212,405,250,NaN,NaN,NaN,NaN,NaN,167
mean,600.500000,1090.850000,NaN,NaN,NaN,NaN,2592.431525,52.627178,3.108333,1.733333,0.067500,NaN
std,346.554469,51.657603,NaN,NaN,NaN,NaN,1634.231457,25.285061,2.898896,1.618973,0.250991,NaN
min,1.000000,1000.000000,NaN,NaN,NaN,NaN,4.600000,0.000000,0.000000,0.000000,0.000000,NaN
25%,300.750000,1046.000000,NaN,NaN,NaN,NaN,1300.060000,30.750000,1.000000,0.000000,0.000000,NaN
50%,600.500000,1091.000000,NaN,NaN,NaN,NaN,2520.040000,52.000000,3.000000,2.000000,0.000000,NaN
75%,900.250000,1134.000000,NaN,NaN,NaN,NaN,3753.340000,74.000000,5.000000,3.000000,0.000000,NaN


In [76]:
df_clean.to_csv("data/cleaned_synthetic_fraud_dataset.csv", index=False)
